In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.gridspec as gs
import regionmask
import xarray as xr
import cartopy.crs as ccrs
import seaborn as sns
from shapely.geometry import box
from shapely.geometry import Polygon
from shapely.geometry import Point

# Load Data

In [ ]:
# load fire observations
fire_obs = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp")

# load study area
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")

# Assign each fire to a country

In [ ]:
#label the country each fire event belongs to
fire_obs["country"] = pd.Series(np.nan, dtype = "str")

In [ ]:
for index, fire in fire_obs.iterrows():
    
    fire_geometry = gpd.GeoSeries(data = [fire.geometry], crs = "EPSG:4326")
    fire_geometry = fire_geometry.to_crs(epsg = 3035)
    
    #get fire centroid in 3035 prj
    fire_centroid = zip(fire_geometry.geometry.centroid.x, fire_geometry.geometry.centroid.y)

    #reproject the centroid to 4326 prj
    gdf_fire_centroid = gpd.GeoDataFrame({'geometry': [Point(fire_centroid)]}, crs = "EPSG:3035")
    gdf_fire_centroid = gdf_fire_centroid.to_crs(epsg = 4326)

    for _, ctr_shp in study_area.iterrows():
        geom = ctr_shp.geometry
        if gdf_fire_centroid.geometry.iloc[0].covered_by(geom): 
            fire_obs.loc[index, "country"] = ctr_shp["ISO3_CODE"]
            break

In [ ]:
fire_obs[fire_obs['country'].isna()]  #AREA_3_91_FS_18_2009_2009_113_1 should belong to Russia(RUS) --> assign manually

In [ ]:
fire_obs.loc[fire_obs['ptch_id'] == 'AREA_3_91_FS_18_2009_2009_113_1', "country"] = 'RUS'

In [ ]:
fire_obs['country'].isna().any()

In [ ]:
# country-level fire event numbers (sum up to 20103)
fire_obs["country"].value_counts().sum()

In [ ]:
# visual check
ctr_list = fire_obs["country"].value_counts().index.tolist()

In [ ]:
# check each region individually
fig, ax = plt.subplots(1, 1, figsize = (5, 9))
ctr = ctr_list[22]  #adapt here

fire_obs_ctr = fire_obs[fire_obs["country"] == ctr]

# reproject 3035
fire_obs_ctr = fire_obs_ctr.to_crs(epsg = 3035)

# get centers
fire_obs_centers = zip(fire_obs_ctr.geometry.centroid.x, fire_obs_ctr.geometry.centroid.y)
fire_obs_centers_gdf = gpd.GeoDataFrame({'geometry': [Point(fire_center) for fire_center in fire_obs_centers]}, crs = "EPSG:3035")
fire_obs_centers_gdf = fire_obs_centers_gdf.to_crs(epsg = 4326)

ctr_shp = study_area[study_area["ISO3_CODE"] == ctr]

study_area.boundary.plot(ax = ax, edgecolor = "#909090", linewidth = 1)
ctr_shp.boundary.plot(ax = ax, edgecolor = "black", linewidth = 1)
fire_obs_centers_gdf.plot(ax = ax, marker = 'o', color = "red", markersize = 0.8)

ax.set_aspect(1.5, adjustable='box')
    
plt.tight_layout()
plt.show()

In [ ]:
fire_obs.to_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region_w_country.shp")